02 Pemeriksaan Duplikasi Korpus

Digunakan untuk memeriksa duplikasi dokumen hasil crawling pada korpus mentah forum diskusi Anime dan Game.

Pemeriksaan duplikasi dilakukan melalui tiga metode:

1. **Duplikasi URL:** memeriksa dokumen yang memiliki URL sama setelah normalisasi sederhana.
2. **Duplikasi hash konten:** memeriksa kesamaan teks berdasarkan nilai hash SHA-256 yang dihitung dari judul dan konten dokumen.
3. **Duplikasi kemiripan teks:** mengidentifikasi pasangan dokumen dengan tingkat kemiripan teks sebesar 95% atau lebih menggunakan TF-IDF character n-gram dan cosine similarity.

Hasil pemeriksaan disimpan di folder `data/audit` sebagai dokumentasi audit. Notebook ini tidak menghapus atau mengubah korpus mentah.

In [1]:
# Mengimpor pustaka yang diperlukan
import json
import hashlib
import re
from pathlib import Path
from collections import defaultdict

import pandas as pd

print("Library berhasil diimpor.")

Library berhasil diimpor.


1. Menentukan Lokasi Proyek

In [2]:
# Menentukan lokasi folder proyek berdasarkan lokasi notebook
ROOT = Path.cwd().resolve()

# Jika notebook dijalankan dari folder crawler,
# folder proyek berada satu tingkat di atasnya
if ROOT.name.lower() == "crawler":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
AUDIT = ROOT / "data" / "audit"

INPUT_FILE = RAW / "korpus.json"

# Membuat folder audit jika belum tersedia
AUDIT.mkdir(parents=True, exist_ok=True)

print("Folder proyek:", ROOT)
print("Lokasi korpus:", INPUT_FILE)
print("Folder audit:", AUDIT)
print("File korpus tersedia:", INPUT_FILE.exists())

Folder proyek: D:\semester 5\penelusuran informasi\searchengine
Lokasi korpus: D:\semester 5\penelusuran informasi\searchengine\data\raw\korpus.json
Folder audit: D:\semester 5\penelusuran informasi\searchengine\data\audit
File korpus tersedia: True


2. Memuat Korpus Mentah

Korpus mentah yang telah dikumpulkan melalui proses crawling dimuat dari file korpus.json.

Data diperiksa untuk memastikan format file dapat dibaca dan jumlah dokumen sesuai dengan hasil crawling sebelumnya.

In [3]:
# Membaca korpus mentah dalam format JSON
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

# Memastikan korpus berbentuk list
if not isinstance(data, list):
    raise ValueError("Format korpus harus berupa list dokumen.")

print("Jumlah dokumen:", len(data))

# Menampilkan satu contoh dokumen untuk melihat struktur datanya
if data:
    print("\nContoh dokumen:")
    print(json.dumps(data[0], ensure_ascii=False, indent=2)[:2000])

Jumlah dokumen: 4096

Contoh dokumen:
{
  "source": "MAL",
  "title": "Upcoming Dubbed Anime - Forums - MyAnimeList.net",
  "content": "Dec 1, 2017 1:59 AM # 1 Kenny_Stryker Offline Mar 2017 1213 I've tried searching for this thread, but couldn't find a similar thread, so I decided to make my own thread to keep a track of upcoming dubbed anime. \n \nSo, if you prefer dub anime, look no further than this.  \n \nLast Updated:  November 23, 2026 \n \n Currently Streaming SimulDubbed Anime - 45 \n Monday \n PSYREN  (Episodes: 5/?) \n Tuesday \n One Piece   (Episodes: 1160/????) \n The Classroom of a Black Cat and a Witch  (Episodes: 1/24) \n The Laid-Off Cheat-Granting Mage Enjoys a Second Lease on Life  (Episodes: 7/12) \n Wednesday \n A Returner's Magic Should Be Special Season 2  (Episodes: 7/12) \n Ingoku Danchi: Deviant's Apartment Complex  (Episodes: 2/12) \n KAMUI: He's Behind You!  (Episodes: 10/12) \n Magic Knight Rayearth  (Episodes: 5/?) \n Nia Liston: The Merciless Maiden  (Epi

3. Pemeriksaan Duplikasi URL

Pemeriksaan URL dilakukan untuk menemukan dokumen yang menggunakan URL identik.

Hasil pemeriksaan digunakan untuk mengetahui jumlah URL yang berulang dalam korpus.

In [4]:
# Fungsi untuk menormalisasi URL
def normalize_url(url):
    if not url:
        return ""

    url = str(url).strip()

    # Menghapus fragment URL, misalnya #comment
    url = url.split("#", 1)[0]

    # Menghapus slash terakhir
    if len(url) > 1:
        url = url.rstrip("/")

    return url


# Mengelompokkan dokumen berdasarkan URL yang telah dinormalisasi
url_groups = defaultdict(list)

for i, doc in enumerate(data):
    url = normalize_url(doc.get("url", ""))

    if url:
        url_groups[url].append(i)


# Mengambil kelompok URL yang muncul lebih dari satu kali
duplicate_url_groups = {
    url: indices
    for url, indices in url_groups.items()
    if len(indices) > 1
}


# Menghitung jumlah dokumen yang terlibat dalam kelompok duplikat
duplicate_url_docs = sum(
    len(indices)
    for indices in duplicate_url_groups.values()
)


print("Jumlah kelompok URL duplikat:", len(duplicate_url_groups))
print("Jumlah dokumen dalam kelompok URL duplikat:", duplicate_url_docs)


# Menampilkan contoh hasil pemeriksaan
for url, indices in list(duplicate_url_groups.items())[:20]:
    print("\nURL:", url)

    for idx in indices:
        print(
            "-",
            data[idx].get("source"),
            "|",
            data[idx].get("title")
        )

Jumlah kelompok URL duplikat: 0
Jumlah dokumen dalam kelompok URL duplikat: 0


4. Pemeriksaan Duplikasi Berdasarkan Hash SHA-256

Pada tahap ini, hash SHA-256 dihitung dari gabungan judul dan konten setiap dokumen.

Hash digunakan untuk mengidentifikasi dokumen yang memiliki teks identik berdasarkan hasil penggabungan judul dan konten. Dokumen yang memiliki nilai hash sama dikelompokkan sebagai duplikasi hash.

Pemeriksaan ini tidak menghapus dokumen, melainkan mencatat hasilnya untuk dianalisis lebih lanjut.

In [5]:
# Fungsi untuk mengambil gabungan judul dan konten dokumen
def get_content(doc):
    title = str(doc.get("title") or "")
    content = str(
        doc.get("content")
        or doc.get("text")
        or doc.get("body")
        or ""
    )

    return (title + "\n" + content).strip()


# Fungsi menghitung hash SHA-256
def calculate_hash(doc):
    text = get_content(doc)

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# Mengelompokkan dokumen berdasarkan hash konten
hash_groups = defaultdict(list)

for i, doc in enumerate(data):
    h = calculate_hash(doc)
    hash_groups[h].append(i)


# Mengambil kelompok hash yang berulang
duplicate_hash_groups = {
    h: indices
    for h, indices in hash_groups.items()
    if len(indices) > 1
}


# Menghitung dokumen yang terlibat dalam kelompok duplikat hash
duplicate_hash_docs = sum(
    len(indices)
    for indices in duplicate_hash_groups.values()
)


print("Jumlah kelompok hash duplikat:", len(duplicate_hash_groups))
print("Jumlah dokumen dalam kelompok hash duplikat:", duplicate_hash_docs)


# Menampilkan contoh hasil
for h, indices in list(duplicate_hash_groups.items())[:20]:
    print("\nHash:", h)

    for idx in indices:
        print(
            "-",
            data[idx].get("source"),
            "|",
            data[idx].get("title"),
            "|",
            data[idx].get("url")
        )

Jumlah kelompok hash duplikat: 2
Jumlah dokumen dalam kelompok hash duplikat: 4

Hash: f8789e931c4c8bd11061857c596f321db45177fb48f6a14c4d2afbcdad821805
- MAL | Uncle's Obsession with Cute Things ⭐ Profile Badge Event - Forums - MyAnimeList.net | https://myanimelist.net/forum/?topicid=2284331&utm_source=MAL&utm_medium=globalmenu_ojikawa
- MAL | Uncle's Obsession with Cute Things ⭐ Profile Badge Event - Forums - MyAnimeList.net | https://myanimelist.net/forum/?topicid=2284331&utm_source=MAL&utm_medium=textribbon_ojikawa

Hash: ca3d7d97a09c8542ad9a325fca493ec15ceb240852162f5a7472d787c64f3d65
- AniList | Forum - Anime & Manga Discussion · AniList | https://anilist.co/forum/new?token=agNasg5dk9jxq7OcXAjbveTdiXuwn0o1INQQPCcU
- AniList | Forum - Anime & Manga Discussion · AniList | https://anilist.co/forum/search


5. Pemeriksaan Kemiripan Teks di Atas 95%

Pemeriksaan kemiripan dilakukan untuk mengidentifikasi dokumen yang memiliki kemiripan teks tinggi, tetapi belum tentu memiliki konten yang identik.

Metode yang digunakan:

- TF-IDF character n-gram (3–5 karakter).
- Cosine similarity.
- Ambang batas kemiripan sebesar 95%.

Character n-gram digunakan agar perbedaan kecil pada teks, seperti perubahan karakter atau tanda baca, tetap dapat dianalisis.

Pasangan dokumen yang memiliki nilai cosine similarity sebesar 0,95 atau lebih akan dicatat sebagai kandidat duplikat.

Pemeriksaan dilakukan secara bertahap untuk mengurangi penggunaan memori.

In [6]:
# Mengimpor metode TF-IDF dan cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Menyiapkan teks dokumen
texts = [get_content(doc) for doc in data]

# Menentukan ambang batas kemiripan
threshold = 0.95

# Menampung hasil pasangan dokumen yang mirip
similarity_pairs = []

# Mengubah teks menjadi representasi TF-IDF character n-gram
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=1
)

matrix = vectorizer.fit_transform(texts)

# Membandingkan dokumen secara bertahap
batch_size = 100

for start in range(0, len(data), batch_size):

    end = min(start + batch_size, len(data))

    # Menghitung cosine similarity antara batch dan seluruh dokumen
    sims = cosine_similarity(
        matrix[start:end],
        matrix
    )

    # Memeriksa setiap dokumen dalam batch
    for local_i in range(end - start):

        global_i = start + local_i

        # Hanya mengambil pasangan dengan indeks lebih besar
        # agar tidak menghasilkan pasangan yang sama dua kali
        for j in range(global_i + 1, len(data)):

            score = float(sims[local_i, j])

            if score >= threshold:

                similarity_pairs.append({
                    "doc_i": global_i,
                    "doc_j": j,
                    "similarity": score,
                    "title_i": data[global_i].get("title", ""),
                    "title_j": data[j].get("title", ""),
                    "url_i": data[global_i].get("url", ""),
                    "url_j": data[j].get("url", ""),
                    "source_i": data[global_i].get("source", ""),
                    "source_j": data[j].get("source", "")
                })

    print(f"Dokumen {start + 1}–{end} selesai diperiksa.")


print("\nJumlah pasangan dengan similarity >= 95%:",
      len(similarity_pairs))

Dokumen 1–100 selesai diperiksa.
Dokumen 101–200 selesai diperiksa.
Dokumen 201–300 selesai diperiksa.
Dokumen 301–400 selesai diperiksa.
Dokumen 401–500 selesai diperiksa.
Dokumen 501–600 selesai diperiksa.
Dokumen 601–700 selesai diperiksa.
Dokumen 701–800 selesai diperiksa.
Dokumen 801–900 selesai diperiksa.
Dokumen 901–1000 selesai diperiksa.
Dokumen 1001–1100 selesai diperiksa.
Dokumen 1101–1200 selesai diperiksa.
Dokumen 1201–1300 selesai diperiksa.
Dokumen 1301–1400 selesai diperiksa.
Dokumen 1401–1500 selesai diperiksa.
Dokumen 1501–1600 selesai diperiksa.
Dokumen 1601–1700 selesai diperiksa.
Dokumen 1701–1800 selesai diperiksa.
Dokumen 1801–1900 selesai diperiksa.
Dokumen 1901–2000 selesai diperiksa.
Dokumen 2001–2100 selesai diperiksa.
Dokumen 2101–2200 selesai diperiksa.
Dokumen 2201–2300 selesai diperiksa.
Dokumen 2301–2400 selesai diperiksa.
Dokumen 2401–2500 selesai diperiksa.
Dokumen 2501–2600 selesai diperiksa.
Dokumen 2601–2700 selesai diperiksa.
Dokumen 2701–2800 sele

6. Menampilkan Hasil Kemiripan

Hasil pemeriksaan ditampilkan dalam bentuk tabel yang berisi indeks dokumen, judul, URL, sumber, dan nilai kemiripan.

Tabel ini digunakan untuk membantu proses peninjauan kandidat dokumen duplikat sebelum proses pembersihan korpus dilakukan.

In [7]:
# Mengubah hasil pasangan kemiripan menjadi DataFrame
similarity_df = pd.DataFrame(similarity_pairs)

if not similarity_df.empty:

    # Mengurutkan hasil dari similarity tertinggi
    similarity_df = similarity_df.sort_values(
        by="similarity",
        ascending=False
    ).reset_index(drop=True)

    display(similarity_df.head(50))

else:

    print("Tidak ditemukan pasangan dokumen dengan similarity >= 95%.")

,doc_i,doc_j,similarity,title_i,title_j,url_i,url_j,source_i,source_j
0,91,93,1.000000,Forum - Anime & Manga Discussion · AniList,Forum - Anime & Manga Discussion · AniList,https://anilist.co/forum/new?token=agNasg5dk9j...,https://anilist.co/forum/search,AniList,AniList
1,15,16,1.000000,Uncle's Obsession with Cute Things ⭐ Profile B...,Uncle's Obsession with Cute Things ⭐ Profile B...,https://myanimelist.net/forum/?topicid=2284331...,https://myanimelist.net/forum/?topicid=2284331...,MAL,MAL
2,59,60,0.997116,Rise of the Tomb Raider General Discussions ::...,Rise of the Tomb Raider General Discussions ::...,https://steamcommunity.com/app/391220/discussi...,https://steamcommunity.com/app/391220/discussi...,Steam,Steam


7. Menyimpan Hasil Pemeriksaan Duplikasi

Hasil pemeriksaan disimpan ke folder `data/audit` agar dapat digunakan sebagai dokumentasi proses crawling dan pemeriksaan kualitas korpus.

File yang dihasilkan:

- `hasil_duplicate_check.json` : ringkasan pemeriksaan URL, hash, dan similarity.
- `pasangan_similarity.csv` : daftar pasangan dokumen dengan similarity sebesar 95% atau lebih.

File korpus mentah tetap dipertahankan dan tidak diubah.

In [8]:
# Menyusun ringkasan hasil pemeriksaan
audit_result = {
    "total_documents": len(data),

    "url_duplicate_groups": len(duplicate_url_groups),
    "documents_in_url_duplicate_groups": duplicate_url_docs,

    "hash_duplicate_groups": len(duplicate_hash_groups),
    "documents_in_hash_duplicate_groups": duplicate_hash_docs,

    "similarity_threshold": threshold,
    "similarity_pairs_count": len(similarity_pairs),
    "similarity_pairs": similarity_pairs
}


# Menyimpan laporan pemeriksaan dalam format JSON
audit_file = AUDIT / "hasil_duplicate_check.json"

with open(audit_file, "w", encoding="utf-8") as f:
    json.dump(
        audit_result,
        f,
        ensure_ascii=False,
        indent=2
    )


# Menyimpan daftar pasangan similarity dalam format CSV
similarity_csv = AUDIT / "pasangan_similarity.csv"

similarity_df.to_csv(
    similarity_csv,
    index=False,
    encoding="utf-8-sig"
)


print("Laporan audit tersimpan:", audit_file)
print("Daftar similarity tersimpan:", similarity_csv)

Laporan audit tersimpan: D:\semester 5\penelusuran informasi\searchengine\data\audit\hasil_duplicate_check.json
Daftar similarity tersimpan: D:\semester 5\penelusuran informasi\searchengine\data\audit\pasangan_similarity.csv


8. Ringkasan Pemeriksaan Duplikasi

Pemeriksaan duplikasi dilakukan terhadap korpus mentah yang diperoleh dari proses crawling.

Pemeriksaan mencakup duplikasi URL, kesamaan hash konten SHA-256, dan kemiripan teks menggunakan cosine similarity dengan ambang batas 95%.

Hasil pemeriksaan disimpan dalam folder audit sebagai dokumentasi evaluasi kualitas korpus.

Hasil pemeriksaan duplikasi tidak secara otomatis menghapus dokumen. Kandidat duplikat perlu ditinjau lebih lanjut sebelum digunakan pada tahap preprocessing dan indexing.

In [9]:
# Menampilkan ringkasan akhir pemeriksaan
print("=" * 50)
print("RINGKASAN PEMERIKSAAN DUPLIKASI")
print("=" * 50)

print("Total dokumen:", len(data))

print("\nPemeriksaan URL:")
print("Kelompok duplikat:", len(duplicate_url_groups))
print("Dokumen terlibat:", duplicate_url_docs)

print("\nPemeriksaan hash:")
print("Kelompok duplikat:", len(duplicate_hash_groups))
print("Dokumen terlibat:", duplicate_hash_docs)

print("\nPemeriksaan similarity:")
print("Ambang batas:", threshold)
print("Pasangan >= 95%:", len(similarity_pairs))

print("\nFile hasil:")
print(audit_file)
print(similarity_csv)

print("\nPemeriksaan selesai.")

RINGKASAN PEMERIKSAAN DUPLIKASI
Total dokumen: 4096

Pemeriksaan URL:
Kelompok duplikat: 0
Dokumen terlibat: 0

Pemeriksaan hash:
Kelompok duplikat: 2
Dokumen terlibat: 4

Pemeriksaan similarity:
Ambang batas: 0.95
Pasangan >= 95%: 3

File hasil:
D:\semester 5\penelusuran informasi\searchengine\data\audit\hasil_duplicate_check.json
D:\semester 5\penelusuran informasi\searchengine\data\audit\pasangan_similarity.csv

Pemeriksaan selesai.
